# how-fast: Benchmark Analysis

Loads results from `results/` and produces comparison charts across experiments and workloads.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams["figure.figsize"] = (12, 5)
matplotlib.rcParams["figure.dpi"] = 100

RESULTS_DIR = Path("../results")

In [ ]:
# Cell 2: Load all summary.json files into a unified DataFrame

rows = []
for summary_file in sorted(RESULTS_DIR.rglob("summary.json")):
    with open(summary_file) as f:
        data = json.load(f)
    for entry in data:
        entry["run_dir"] = str(summary_file.parent)
        rows.append(entry)

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} rows from {df['experiment'].nunique()} experiments")
df

In [ ]:
# Cell 3: Per-workload comparison tables — gateway vs direct rows

from IPython.display import display, HTML

metric_cols = [
    "experiment",
    "n_requests", "n_errors", "error_rate",
    "total_latency_p50_s", "total_latency_p95_s",
    "ttft_p50_s", "ttft_p95_s",
    "tps_p50", "tps_p95",
    "cost_per_request_usd",
]
display_cols = [c for c in metric_cols if c in df.columns]

for workload in sorted(df["workload"].unique()):
    wdf = (
        df[df["workload"] == workload][display_cols]
        .set_index("experiment")
        .sort_index()
    )
    display(HTML(f"<h3>Workload: {workload}</h3>"))
    display(wdf)


In [ ]:
# Cell 4: Latency comparison — total_latency_p95 by experiment, grouped by workload

pivot = df.pivot_table(index="workload", columns="experiment", values="total_latency_p95_s")
pivot.plot(kind="bar", title="Total Latency P95 (s) by Experiment")
plt.ylabel("Latency P95 (s)")
plt.xticks(rotation=0)
plt.legend(title="Experiment")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: TTFT comparison — ttft_p95 by experiment

if "ttft_p95_s" in df.columns and df["ttft_p95_s"].notna().any():
    pivot = df.pivot_table(index="workload", columns="experiment", values="ttft_p95_s")
    pivot.plot(kind="bar", title="TTFT P95 (s) by Experiment")
    plt.ylabel("TTFT P95 (s)")
    plt.xticks(rotation=0)
    plt.legend(title="Experiment")
    plt.tight_layout()
    plt.show()
else:
    print("No TTFT data available (non-streaming mode?)")

In [ ]:
# Cell 6: Throughput comparison — tps_p50 by experiment

pivot = df.pivot_table(index="workload", columns="experiment", values="tps_p50")
pivot.plot(kind="bar", title="Throughput P50 (tokens/s) by Experiment")
plt.ylabel("Tokens/s P50")
plt.xticks(rotation=0)
plt.legend(title="Experiment")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Latency vs Throughput scatter

fig, ax = plt.subplots()
for exp, group in df.groupby("experiment"):
    ax.scatter(group["tps_p50"], group["total_latency_p95_s"], label=exp, s=80)
    for _, row in group.iterrows():
        ax.annotate(row["workload"], (row["tps_p50"], row["total_latency_p95_s"]),
                    fontsize=8, alpha=0.7)
ax.set_xlabel("Throughput P50 (tokens/s)")
ax.set_ylabel("Latency P95 (s)")
ax.set_title("Latency vs Throughput")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Cost vs Performance scatter

if "cost_per_request_usd" in df.columns and df["cost_per_request_usd"].notna().any():
    fig, ax = plt.subplots()
    for exp, group in df.groupby("experiment"):
        ax.scatter(group["tps_p50"], group["cost_per_request_usd"], label=exp, s=80)
    ax.set_xlabel("Throughput P50 (tokens/s)")
    ax.set_ylabel("Cost per Request (USD)")
    ax.set_title("Cost vs Performance")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No cost data available")

In [ ]:
# Cell 9: GPU utilization over time (from gpu_metrics.csv)

gpu_rows = []
for csv_file in sorted(RESULTS_DIR.rglob("gpu_metrics.csv")):
    gpu_df = pd.read_csv(csv_file)
    gpu_rows.append(gpu_df)

if gpu_rows:
    all_gpu = pd.concat(gpu_rows, ignore_index=True)
    all_gpu["timestamp_utc"] = pd.to_datetime(all_gpu["timestamp_utc"])
    fig, ax = plt.subplots()
    for exp, group in all_gpu.groupby("experiment"):
        group = group.sort_values("timestamp_utc")
        elapsed = (group["timestamp_utc"] - group["timestamp_utc"].iloc[0]).dt.total_seconds()
        ax.plot(elapsed, group["gpu_util_pct"], label=exp)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("GPU Utilization %")
    ax.set_title("GPU Utilization Over Time")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No GPU metrics data found")

In [ ]:
# Cell 10: Gateway overhead — direct vs gateway latency per workload

direct_df = df[df["experiment"].str.endswith("_direct")]
gateway_df = df[~df["experiment"].str.endswith("_direct")]

if not direct_df.empty:
    workload_list = sorted(df["workload"].unique())
    fig, axes = plt.subplots(1, len(workload_list), figsize=(14, 5), sharey=True)
    if not hasattr(axes, "__iter__"):
        axes = [axes]

    for ax, workload in zip(axes, workload_list):
        d_lat = direct_df[direct_df["workload"] == workload].set_index("experiment")["total_latency_p50_s"]
        g_lat = gateway_df[gateway_df["workload"] == workload].set_index("experiment")["total_latency_p50_s"]

        combined = pd.concat([
            d_lat.rename(lambda x: x.replace("_direct", "") + "\n(direct)"),
            g_lat.rename(lambda x: x + "\n(gateway)"),
        ])
        combined.plot(kind="bar", ax=ax, title=workload, color=["steelblue", "coral"])
        ax.set_ylabel("Latency P50 (s)" if ax == axes[0] else "")
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=0)

    fig.suptitle("Direct vs Gateway Latency P50 per Workload")
    plt.tight_layout()
    plt.show()
else:
    print("No direct baseline data — run with --include-direct to enable.")


In [ ]:
# Cell 11: Error analysis — stacked bar of error types by experiment

req_rows = []
for csv_file in sorted(RESULTS_DIR.rglob("requests.csv")):
    req_df = pd.read_csv(csv_file)
    req_rows.append(req_df)

if req_rows:
    all_req = pd.concat(req_rows, ignore_index=True)
    errors = all_req[all_req["status"] != "ok"]
    if not errors.empty:
        ct = errors.groupby(["experiment", "status"]).size().unstack(fill_value=0)
        ct.plot(kind="bar", stacked=True, title="Error Types by Experiment")
        plt.ylabel("Error Count")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("No errors found — all requests succeeded!")
else:
    print("No request CSV data found")

In [ ]:
# Cell 12: SLO validation table

slo_rows = []
for slo_file in sorted(RESULTS_DIR.rglob("slo_report.json")):
    with open(slo_file) as f:
        data = json.load(f)
    slo_rows.extend(data)

if slo_rows:
    slo_df = pd.DataFrame(slo_rows)
    # Pivot: rows = (experiment, workload), columns = SLO name, values = pass/fail
    slo_pivot = slo_df.pivot_table(
        index=["experiment", "workload"],
        columns="slo",
        values="pass",
        aggfunc="first",
    )

    def color_bool(val):
        if val is True:
            return "background-color: #90EE90"
        elif val is False:
            return "background-color: #FFB6C1"
        return ""

    display(slo_pivot.style.map(color_bool))
else:
    print("No SLO report data found. Run benchmarks with SLO config first.")

In [ ]:
# Cell 13: SLO radar chart — normalized compliance per experiment

import numpy as np

if slo_rows:
    slo_df = pd.DataFrame(slo_rows)
    # Compute pass rate per (experiment, slo)
    pass_rates = slo_df.groupby(["experiment", "slo"])["pass"].mean().unstack(fill_value=0)

    categories = list(pass_rates.columns)
    N = len(categories)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]  # close the polygon

    fig, ax = plt.subplots(subplot_kw=dict(polar=True), figsize=(8, 8))
    for exp in pass_rates.index:
        vals = pass_rates.loc[exp].tolist()
        vals += vals[:1]
        ax.plot(angles, vals, label=exp, linewidth=2)
        ax.fill(angles, vals, alpha=0.1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=8)
    ax.set_ylim(0, 1)
    ax.set_title("SLO Compliance by Experiment", y=1.08)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.show()
else:
    print("No SLO data for radar chart")